In [1]:
import pandas as pd
import numpy as np
from scipy.stats import poisson

team_stats = pd.read_csv('../data/team_stats.csv')

def predict_match(home_team, away_team, team_stats, max_goals=6):
    home = team_stats[team_stats['team'] == home_team].iloc[0]
    away = team_stats[team_stats['team'] == away_team].iloc[0]
    
    home_expected_goals = np.clip(
        home['home_goals_scored'] * away['away_goals_conceded'] / 0.9, 0.3, 4.0)
    away_expected_goals = np.clip(
        away['away_goals_scored'] * home['home_goals_conceded'] / 1.1, 0.3, 4.0)
    
    home_probs = [poisson.pmf(i, home_expected_goals) for i in range(max_goals+1)]
    away_probs = [poisson.pmf(i, away_expected_goals) for i in range(max_goals+1)]
    score_matrix = np.outer(home_probs, away_probs)
    
    home_win_prob = np.tril(score_matrix, -1).sum()
    draw_prob     = np.trace(score_matrix)
    away_win_prob = np.triu(score_matrix, 1).sum()
    
    max_idx = np.unravel_index(score_matrix.argmax(), score_matrix.shape)
    
    if home_win_prob >= away_win_prob and home_win_prob >= draw_prob:
        winner = home_team
    else:
        winner = away_team
    
    # In knockout, no draws allowed, so pick a winner
    # If draw is most likely, winner is whoever has higher win prob
    
    # Penalty probability, roughly 20% of knockout matches go to pens
    penalties = np.random.random() < 0.20
    
    return {
        'home_team': home_team,
        'away_team': away_team,
        'predicted_home_goals': int(max_idx[0]),
        'predicted_away_goals': int(max_idx[1]),
        'home_win_prob': round(home_win_prob * 100, 1),
        'draw_prob': round(draw_prob * 100, 1),
        'away_win_prob': round(away_win_prob * 100, 1),
        'winner': winner,
        'penalties': penalties
    }

def predict_extras(home_team, away_team, team_stats):
    home = team_stats[team_stats['team'] == home_team].iloc[0]
    away = team_stats[team_stats['team'] == away_team].iloc[0]
    rank_factor = (home['fifa_rank'] + away['fifa_rank']) / 200
    return {
        'corners': int(round(10.2 + rank_factor * 2, 0)),
        'yellow_cards': int(round(3.1 + rank_factor * 0.5, 0)),
        'red_cards': 0
    }

print("Functions ready")

Functions ready


In [11]:
group_winners = {
    'A': 'Mexico',    
    'B': 'Canada',
    'C': 'Brazil',
    'D': 'United States',
    'E': 'Germany',
    'F': 'Japan',
    'G': 'Belgium',
    'H': 'Spain',
    'I': 'France',
    'J': 'Argentina',
    'K': 'Portugal',
    'L': 'England',
}

group_runners = {
    'A': 'Morocco',
    'B': 'Japan',
    'C': 'Ghana',
    'D': 'South Korea',
    'E': 'Australia',
    'F': 'Colombia',
    'G': 'Senegal',
    'H': 'Uruguay',
    'I': 'Paraguay',
    'J': 'Argentina',
    'K': 'United States',
    'L': 'Austria',
}

# Best 8 third place teams
best_thirds = [
    'Iran', 'Senegal', 'Czechia', 'Türkiye',
    'Cabo Verde', 'Austria', 'Ghana', 'Bosnia and Herzegovina'
]

print("Teams loaded")
print(f"Group winners: {len(group_winners)}")
print(f"Group runners-up: {len(group_runners)}")
print(f"Best thirds: {len(best_thirds)}")

Teams loaded
Group winners: 12
Group runners-up: 12
Best thirds: 8


In [12]:
# Official 2026 bracket matchup pattern
# Winners vs best thirds, runners vs runners etc.
round_of_32 = [
    (group_winners['A'],  group_runners['B']),
    (group_winners['C'],  group_runners['D']),
    (group_winners['E'],  group_runners['F']),
    (group_winners['G'],  group_runners['H']),
    (group_winners['I'],  group_runners['J']),
    (group_winners['K'],  group_runners['L']),
    (group_winners['B'],  group_runners['A']),
    (group_winners['D'],  group_runners['C']),
    (group_winners['F'],  group_runners['E']),
    (group_winners['H'],  group_runners['G']),
    (group_winners['J'],  group_runners['I']),
    (group_winners['L'],  group_runners['K']),
    (group_winners['A'],  best_thirds[0]),
    (group_winners['E'],  best_thirds[1]),
    (group_winners['I'],  best_thirds[2]),
    (group_winners['C'],  best_thirds[3]),
]

print("Round of 32 matchups:")
for i, (home, away) in enumerate(round_of_32, 1):
    print(f"  Match {i}: {home} vs {away}")

Round of 32 matchups:
  Match 1: Mexico vs Japan
  Match 2: Brazil vs South Korea
  Match 3: Germany vs Colombia
  Match 4: Belgium vs Uruguay
  Match 5: France vs Argentina
  Match 6: Portugal vs Austria
  Match 7: Canada vs Morocco
  Match 8: United States vs Ghana
  Match 9: Japan vs Australia
  Match 10: Spain vs Senegal
  Match 11: Argentina vs Paraguay
  Match 12: England vs United States
  Match 13: Mexico vs Iran
  Match 14: Germany vs Senegal
  Match 15: France vs Czechia
  Match 16: Brazil vs Türkiye


In [13]:
def simulate_round(matchups, round_name, team_stats, multiplier):
    """
    Simulate one round of knockout matches.
    Returns results and list of winners.
    """
    results = []
    winners = []
    
    print(f"\n{'='*50}")
    print(f"  {round_name.upper()}  (×{multiplier} points)")
    print(f"{'='*50}")
    
    for home, away in matchups:
        pred   = predict_match(home, away, team_stats)
        extras = predict_extras(home, away, team_stats)
        winner = pred['winner']
        winners.append(winner)
        
        score = f"{pred['predicted_home_goals']}-{pred['predicted_away_goals']}"
        pens  = " (pens)" if pred['penalties'] else ""
        
        print(f"  {home:<22} vs {away:<22} → {score}{pens} {winner}")
        
        results.append({
            'round': round_name,
            'multiplier': multiplier,
            'home_team': home,
            'away_team': away,
            'home_goals': pred['predicted_home_goals'],
            'away_goals': pred['predicted_away_goals'],
            'winner': winner,
            'penalties': pred['penalties'],
            'corners': extras['corners'],
            'yellow_cards': extras['yellow_cards'],
            'red_cards': extras['red_cards'],
        })
    
    return results, winners

# Test with Round of 32
r32_results, r32_winners = simulate_round(
    round_of_32, 'Round of 32', team_stats, multiplier=1
)
print(f"\n{len(r32_winners)} teams advance to Round of 16")


  ROUND OF 32  (×1 points)
  Mexico                 vs Japan                  → 1-1 Japan
  Brazil                 vs South Korea            → 2-0 Brazil
  Germany                vs Colombia               → 2-1 Germany
  Belgium                vs Uruguay                → 1-1 Belgium
  France                 vs Argentina              → 2-0 (pens) France
  Portugal               vs Austria                → 3-1 Portugal
  Canada                 vs Morocco                → 2-1 Canada
  United States          vs Ghana                  → 3-0 (pens) United States
  Japan                  vs Australia              → 3-0 Japan
  Spain                  vs Senegal                → 3-1 (pens) Spain
  Argentina              vs Paraguay               → 2-0 (pens) Argentina
  England                vs United States          → 2-0 England
  Mexico                 vs Iran                   → 2-1 Mexico
  Germany                vs Senegal                → 3-1 Germany
  France                 vs Czechia

In [14]:
all_knockout_results = []

# Round of 32
r32_results, r32_winners = simulate_round(
    round_of_32, 'Round of 32', team_stats, multiplier=1
)
all_knockout_results.extend(r32_results)

# Round of 16, pair up winners
r16_matchups = [(r32_winners[i], r32_winners[i+1]) 
                for i in range(0, len(r32_winners), 2)]
r16_results, r16_winners = simulate_round(
    r16_matchups, 'Round of 16', team_stats, multiplier=2
)
all_knockout_results.extend(r16_results)

# Quarter-finals
qf_matchups = [(r16_winners[i], r16_winners[i+1]) 
               for i in range(0, len(r16_winners), 2)]
qf_results, qf_winners = simulate_round(
    qf_matchups, 'Quarter-final', team_stats, multiplier=4
)
all_knockout_results.extend(qf_results)

# Semi-finals
sf_matchups = [(qf_winners[i], qf_winners[i+1]) 
               for i in range(0, len(qf_winners), 2)]
sf_results, sf_winners = simulate_round(
    sf_matchups, 'Semi-final', team_stats, multiplier=8
)
all_knockout_results.extend(sf_results)

# Third place playoff
third_place_teams = [r for r in qf_winners if r not in sf_winners]
if len(third_place_teams) >= 2:
    tp_results, tp_winners = simulate_round(
        [(third_place_teams[0], third_place_teams[1])],
        'Third-place playoff', team_stats, multiplier=8
    )
    all_knockout_results.extend(tp_results)

# THE FINAL 
final_result, final_winner = simulate_round(
    [(sf_winners[0], sf_winners[1])],
    'FINAL', team_stats, multiplier=16
)
all_knockout_results.extend(final_result)

print(f"\n{'-'*20}")
print(f"  FIFA WORLD CUP 2026 WINNER: {final_winner[0].upper()}")
print(f"{'-'*20}")


  ROUND OF 32  (×1 points)
  Mexico                 vs Japan                  → 1-1 Japan
  Brazil                 vs South Korea            → 2-0 Brazil
  Germany                vs Colombia               → 2-1 Germany
  Belgium                vs Uruguay                → 1-1 (pens) Belgium
  France                 vs Argentina              → 2-0 France
  Portugal               vs Austria                → 3-1 Portugal
  Canada                 vs Morocco                → 2-1 Canada
  United States          vs Ghana                  → 3-0 United States
  Japan                  vs Australia              → 3-0 Japan
  Spain                  vs Senegal                → 3-1 Spain
  Argentina              vs Paraguay               → 2-0 Argentina
  England                vs United States          → 2-0 England
  Mexico                 vs Iran                   → 2-1 Mexico
  Germany                vs Senegal                → 3-1 (pens) Germany
  France                 vs Czechia              

In [15]:
knockout_df = pd.DataFrame(all_knockout_results)
knockout_df.to_csv('../outputs/knockout_predictions.csv', index=False)

print("\nKnockout predictions saved!")
print(f"Total knockout matches: {len(knockout_df)}")
print("\nSummary by round:")
print(knockout_df.groupby('round')['winner'].count().to_string())


Knockout predictions saved!
Total knockout matches: 31

Summary by round:
round
FINAL             1
Quarter-final     4
Round of 16       8
Round of 32      16
Semi-final        2
